# VeriFin paper metrics

This notebook is the compact, executable audit companion for the paper. It
uses the complete 600-question XBRLBench and 67-question FinanceBench
populations, recomputes outcomes from per-question results, and applies only
candidate-specific manual adjudications. `scripts/paper_scoring.py` is the
canonical implementation.

## Load the locked policy and artifacts

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.paper_scoring import (
    load_jsonl,
    load_manual_adjudications,
    load_policy,
    score_all,
    validate_dataset_ids,
)

policy_path = ROOT / "results/scoring_policy.json"
if not policy_path.is_file():
    policy_path = ROOT / "publication_assets_source/results/scoring_policy.json"
policy = load_policy(policy_path)
adjudications = load_manual_adjudications(
    policy_path.parent / policy["candidate_correctness"]["manual_adjudications"]
)
scored = pd.DataFrame(score_all(ROOT, policy, adjudications))
scored

## Confirm dataset grain and denominators

In [ ]:
dataset_profile = []
for dataset_key, spec in policy["datasets"].items():
    rows = load_jsonl(ROOT / spec["source_path"])
    ids = validate_dataset_ids(rows, dataset_key, spec)
    dataset_profile.append({
        "dataset": dataset_key,
        "rows": len(rows),
        "unique_ids": len(ids),
        "denominator": spec["denominator"],
    })
dataset_profile = pd.DataFrame(dataset_profile)
assert dataset_profile["rows"].tolist() == [600, 67]
assert (dataset_profile["rows"] == dataset_profile["unique_ids"]).all()
dataset_profile

## Recompute the paper outcome table

In [ ]:
columns = ["dataset", "group", "run", "TA", "FA", "TR", "FR", "AB"]
outcomes = scored[columns].copy()
available = outcomes.dropna(subset=["TA"]).copy()
assert ((available[["TA", "FA", "TR", "FR", "AB"]].sum(axis=1))
        == available["dataset"].map({"xbrlfiling": 600, "financebench": 67})).all()

metrics_path = ROOT / "results/paper_metrics.csv"
if not metrics_path.is_file():
    # The private working tree keeps release-only support files in the
    # publication staging area; a built public release places this at results/.
    metrics_path = ROOT / "publication_assets_source/results/paper_metrics.csv"
frozen = pd.read_csv(metrics_path)
frozen_available = frozen.dropna(subset=["TA"]).copy()
reconciled = available.merge(
    frozen_available,
    on=["dataset", "group", "run", "TA", "FA", "TR", "FR", "AB"],
    how="outer",
    indicator=True,
)
assert (reconciled["_merge"] == "both").all()
assert ((frozen["dataset"] == "financebench")
        & (frozen["run"] == "gpt-5.5")
        & frozen["TA"].isna()).sum() == 1
frozen

## Headline safety comparison

In [ ]:
headline = frozen[frozen["group"].isin(["answer_models", "baselines"])].copy()
verifin_available = headline[headline["group"] == "answer_models"].dropna(subset=["FA"])
assert (verifin_available["FA"] == 0).all()

headline[[
    "dataset", "group", "run", "n", "TA", "FA", "TR", "FR", "AB",
    "precision_pct", "decision_accuracy_pct", "coverage_pct",
]].reset_index(drop=True)

## Candidate-specific adjudication audit

The entries below normalize units or composite targets. They do **not** make an
arbitrary candidate correct merely because its question ID is listed. The
AMCOR restructuring question admits both the literal gold percentage and the
paper's grounded-dollar normalization; its caveat should remain visible.

In [ ]:
manual_document = json.loads(
    (policy_path.parent / policy["candidate_correctness"]["manual_adjudications"])
    .read_text(encoding="utf-8")
)
manual_review = pd.DataFrame([
    {
        "id": row["id"],
        "allowed_targets": ", ".join(
            f"{target['value']:g} {target['unit']}" for target in row["allowed_targets"]
        ),
        "reason": row["reason"],
        "caveat": row.get("caveat", ""),
    }
    for row in manual_document["records"]
])
manual_review

## Known artifact gaps

In [ ]:
gaps = pd.DataFrame([
    {
        "artifact": "FinanceBench / GPT-5.5",
        "status": "missing",
        "publication treatment": "Do not infer or manually enter per-question results.",
    },
    {
        "artifact": "Fin-o1 checkpoint identity",
        "status": "not embedded in result files",
        "publication treatment": "Use identity-pending label until the immutable checkpoint is recovered.",
    },
    {
        "artifact": "XBRLBench annotation license",
        "status": "author choice pending",
        "publication treatment": "Add the chosen dataset license before public redistribution.",
    },
])
gaps